# 📄 Forensic Legal Document Processing Pipeline

**Modular Forensic Legal Document Processing Infrastructure**  
Runs entirely in Google Colab — accessible on iPad via Safari.

---
### Architecture
```
Source Documents & Evidence
  └─► [Module 1] File Ingestion (PDFs, images, private files)
        └─► [Module 2] Chunked OCR Engine (pdf[0], pdf[1], ...)
              ├─ Forensic Hashing (SHA-256 chain of custody)
              ├─ Metadata Extraction (PDF/EXIF)
              └─► Analysis & Timeline Mapping (RAG + Vertex AI)
                    ├─ Case Knowledge Base
                    ├─ Legal Ethics Module (ND civil procedure)
                    └─ Timeline Mapper
```
---
### iPad Quick-Start
1. Open this notebook at **[colab.research.google.com](https://colab.research.google.com)** in Safari
2. Tap **▶ Run all** (or run cells one-by-one)
3. Upload documents from iPad **Files** or **Photos** app using the upload widget
4. Results auto-save to **Google Drive** (view in the Drive app on iPad)

> **Student Developer Program:** This notebook uses only services available on the  
> Google Cloud Free Tier / Education coupon: Vision API, Cloud Storage, Vertex AI (Gemini Flash).

## ⚙️ Cell 1 – Setup & Configuration
**Edit the values below to match your GCP project.**

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURE YOUR PROJECT HERE
# ─────────────────────────────────────────────
PROJECT_ID   = "YOUR_PROJECT_ID"    # GCP project ID
BUCKET_NAME  = "YOUR_BUCKET_NAME"   # Cloud Storage bucket
CASE_NAME    = "my_case"            # Short identifier for this legal matter

# Document AI (optional – Vision API is used if left blank)
PROCESSOR_ID = ""                   # e.g. "abc123def456"

# Vertex AI location (for RAG + Gemini)
VERTEX_LOCATION = "us-central1"

# Output folder name in Google Drive
DRIVE_FOLDER = "DocumentPipelineResults"
# ─────────────────────────────────────────────

## 📦 Cell 2 – Install Dependencies
Installs all required packages. Run once per Colab session.

In [ ]:
import subprocess, sys

print("Installing dependencies...")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "google-cloud-vision",
    "google-cloud-storage",
    "google-cloud-documentai",
    "google-cloud-aiplatform",
    "vertexai",
    "PyMuPDF",
    "Pillow",
    "pandas",
    "openpyxl",
    "pyyaml",
    "ipywidgets",
])
print("✅ Dependencies installed.")

## 📁 Cell 3 – Clone Repository & Add to Path
Imports the pipeline modules from GitHub.

In [ ]:
import sys, os

REPO_URL = "https://github.com/carlymariec/document-processing-pipeline.git"
REPO_DIR = "/content/document-processing-pipeline"

if not os.path.exists(REPO_DIR):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    print("Repository already present – pulling latest changes...")
    os.system(f"cd {REPO_DIR} && git pull")

# Add repository to Python path (matches the diagram's sys.path.append('repository'))
sys.path.insert(0, REPO_DIR)
print(f"✅ Repository path added: {REPO_DIR}")

## 🔐 Cell 4 – Google Authentication
Authenticates with Google Cloud using your Google account.  
On iPad: a browser popup will appear — sign in with your Google account.

In [ ]:
from google.colab import auth
from google.colab import drive

# Authenticate with Google Cloud
print("Authenticating with Google Cloud...")
auth.authenticate_user()
print("✅ Google Cloud authentication complete.")

# Mount Google Drive (for saving results – accessible via Drive app on iPad)
print("\nMounting Google Drive...")
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

# Set GCP project
import subprocess
subprocess.run(["gcloud", "config", "set", "project", PROJECT_ID], capture_output=True)
print(f"✅ GCP project set to: {PROJECT_ID}")

## 🔧 Cell 5 – Initialise Pipeline
Creates the pipeline instance with all modules.

In [ ]:
from src.pipeline import DocumentPipeline

pipeline = DocumentPipeline(
    project_id=PROJECT_ID,
    bucket_name=BUCKET_NAME,
    case_name=CASE_NAME,
    vertex_location=VERTEX_LOCATION,
    processor_id=PROCESSOR_ID if PROCESSOR_ID else None,
    use_document_ai=bool(PROCESSOR_ID),
    use_vision_api=True,
    use_vertex_ai=True,
    save_to_drive=True,
    drive_folder=DRIVE_FOLDER,
)

print("✅ Pipeline initialised")
print(f"   Case     : {CASE_NAME}")
print(f"   Project  : {PROJECT_ID}")
print(f"   Bucket   : {BUCKET_NAME}")
print(f"   OCR Mode : {'Document AI' if PROCESSOR_ID else 'Vision API'}")
print(f"   RAG      : Vertex AI Gemini ({VERTEX_LOCATION})")

## 📤 Cell 6 – Upload & Process Documents
Upload files directly from your iPad (Files app, Photos app, or Google Drive).  
Supported: PDF, PNG, JPG, JPEG, TIFF, BMP, GIF, WEBP, TXT, RM

In [ ]:
from google.colab import files as colab_files
import os

print("📎 Tap 'Choose Files' to upload from your iPad...")
print("   (Supports: PDF, PNG, JPG, JPEG, TIFF, TXT, and more)")
print()

uploaded = colab_files.upload()  # iPad: shows Files/Photos picker

if not uploaded:
    print("No files uploaded.")
else:
    print(f"\n✅ {len(uploaded)} file(s) uploaded. Ready to process.")
    for name in uploaded:
        print(f"   • {name} ({len(uploaded[name]) / 1024:.1f} KB)")

## ⚙️ Cell 7 – Run the Forensic Pipeline
Runs all modules on each uploaded file:
- Module 1: File Ingestion
- Module 2: Chunked OCR (page-by-page)
- Forensic Hashing + Metadata
- Legal Ethics Review (ND civil procedure)
- Timeline Event Extraction

In [ ]:
import json
from pathlib import Path

all_results = []

for filename, file_bytes in uploaded.items():
    # Save uploaded bytes to /content/
    local_path = Path("/content") / filename
    local_path.write_bytes(file_bytes)

    # Run the full forensic pipeline
    result = pipeline.run(
        file_path=local_path,
        upload_to_gcs=bool(BUCKET_NAME),   # Skip GCS if bucket not configured
        output_formats=["json", "txt"],
        run_ethics=True,
        run_timeline=True,
        add_to_knowledge_base=True,
    )
    all_results.append(result)

print(f"\n{'═'*55}")
print(f"✅ All {len(all_results)} document(s) processed.")
print(f"{'═'*55}")

## 📊 Cell 8 – View Results
Review OCR text, ethics flags, and timeline for the last processed document.

In [ ]:
from IPython.display import display, Markdown, HTML

if not all_results:
    print("No results yet – run Cell 7 first.")
else:
    result = all_results[-1]  # Show the most recent document

    # ── Document Info ───────────────────────────────────────────────
    display(Markdown(f"### 📄 {result['ingested_file']['original_name']}"))
    display(Markdown(
        f"| Field | Value |\n|---|---|\n"
        f"| Size | {result['ingested_file']['size_mb']:.2f} MB |\n"
        f"| SHA-256 | `{result['custody']['source_sha256'][:32]}…` |\n"
        f"| Pages | {len(result['extraction']['pages'])} |\n"
        f"| OCR Confidence | {result['extraction']['confidence']:.1%} |\n"
        f"| Words | {result['text_analysis']['word_count']} |\n"
        f"| Processed | {result['processed_at']} |"
    ))

    # ── Ethics Report ────────────────────────────────────────────────
    ethics = result.get('ethics_review', {})
    risk = ethics.get('risk_level', 'N/A')
    risk_colour = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🟠', 'CRITICAL': '🔴'}.get(risk, '⚪')
    display(Markdown(f"\n### {risk_colour} Legal Ethics Review – Risk Level: **{risk}**"))
    if ethics.get('summary'):
        print(ethics['summary'])

    # ── ND Compliance ─────────────────────────────────────────────────
    nd = ethics.get('nd_compliance', [])
    if nd:
        display(Markdown("\n**North Dakota Civil Procedure Compliance:**"))
        for item in nd:
            severity_icon = {'CRITICAL': '🔴', 'HIGH': '🟠', 'MEDIUM': '🟡', 'LOW': '🟢'}.get(item.get('severity',''), '⚪')
            print(f"  {severity_icon} [{item['rule']}] {item['finding']}")

    # ── Timeline Events ───────────────────────────────────────────────
    events = result.get('timeline_events', [])
    if events:
        display(Markdown(f"\n### 📅 Timeline Events ({len(events)} found)"))
        for e in sorted(events, key=lambda x: x.get('date_iso','9999'))[:10]:
            print(f"  {e.get('date_iso','?')} [{e.get('event_type','event')}] {e.get('context','')[:100]}")

    # ── OCR Text Preview ──────────────────────────────────────────────
    full_text = result['extraction'].get('full_text', '')
    display(Markdown("\n### 📝 Extracted Text (first 2000 chars)"))
    print(full_text[:2000])

## 🤖 Cell 9 – Ask Questions (RAG + Vertex AI Gemini)
Query all processed documents using natural language.  
Answers are grounded in the actual document content.

In [ ]:
from IPython.display import display, Markdown

# ── Edit your question here ─────────────────────────────────────────
QUESTION = "What are the key dates and deadlines in these documents?"
# ───────────────────────────────────────────────────────────────────

print(f"❓ Question: {QUESTION}\n")
print("Querying Case Knowledge Base + Vertex AI Gemini...\n")

rag_result = pipeline.query(QUESTION)

display(Markdown("### 💬 Answer"))
print(rag_result['answer'])

display(Markdown("\n### 📚 Sources Used"))
for src in rag_result['sources']:
    print(f"  • {src['document']} (page {src['page']}) — relevance: {src['relevance_score']:.3f}")

## 📋 Cell 10 – Generate Case Summary

In [ ]:
from IPython.display import display, Markdown

print("Generating AI case summary with Vertex AI Gemini...\n")
summary = pipeline.summarize_case()

display(Markdown("### 📋 Case Summary"))
print(summary)

## 📅 Cell 11 – Export Full Case Timeline
Exports a Markdown timeline of all events from all documents.  
Automatically saved to Google Drive → visible in Drive app on iPad.

In [ ]:
from IPython.display import display, Markdown

timeline_md = pipeline.export_timeline()

display(Markdown("### 📅 Case Timeline"))
display(Markdown(timeline_md))

print("\n✅ Timeline saved to Google Drive.")
print(f"   Open the Google Drive app on iPad → '{DRIVE_FOLDER}' folder")

## 💾 Cell 12 – Save Knowledge Base & Download Results
Persists the Case Knowledge Base to Google Drive.  
Also lets you download files directly to your iPad.

In [ ]:
from google.colab import files as colab_files
import glob, os

# Save knowledge base to Drive
kb_path = pipeline.save_knowledge_base()
print(f"\n✅ Knowledge base saved: {kb_path}")

# List all result files
result_files = glob.glob("/tmp/pipeline_*/*")
if result_files:
    print(f"\n📁 Result files available for download ({len(result_files)} files):")
    for f in result_files:
        size_kb = os.path.getsize(f) / 1024
        print(f"  • {os.path.basename(f)} ({size_kb:.1f} KB)")

    print("\nDownloading to iPad...")
    for f in result_files:
        colab_files.download(f)
else:
    print("\nNo result files found yet – run Cell 7 first.")

## 📂 Cell 13 – Process File Directly from Google Drive
If you saved a file to the Google Drive app on your iPad, process it here  
without needing to re-upload.

In [ ]:
import os

# ── Set the path to your file in Google Drive ────────────────────────
DRIVE_FILE_PATH = "/content/drive/MyDrive/my_document.pdf"
# ─────────────────────────────────────────────────────────────────────

if not os.path.exists(DRIVE_FILE_PATH):
    print(f"File not found: {DRIVE_FILE_PATH}")
    print("Make sure the file is in your Google Drive and the path is correct.")
    print("\nYour Drive root contents:")
    for f in os.listdir("/content/drive/MyDrive")[:20]:
        print(f"  {f}")
else:
    print(f"Processing from Drive: {DRIVE_FILE_PATH}")
    result = pipeline.run(
        file_path=DRIVE_FILE_PATH,
        run_ethics=True,
        run_timeline=True,
        add_to_knowledge_base=True,
    )
    all_results.append(result)
    print(f"\n✅ Done! Words extracted: {result['text_analysis']['word_count']}")

## 📊 Cell 14 – Knowledge Base Statistics

In [ ]:
from IPython.display import display, Markdown
import pandas as pd

kb_summary = pipeline.knowledge_base.summary()

display(Markdown("### 🗄️ Case Knowledge Base"))
display(Markdown(
    f"| Metric | Value |\n|---|---|\n"
    f"| Case Name | {kb_summary['case_name']} |\n"
    f"| Total Chunks | {kb_summary['total_chunks']} |\n"
    f"| Total Words | {kb_summary['total_words']:,} |\n"
    f"| Documents | {len(kb_summary['documents'])} |"
))

if kb_summary['documents']:
    display(Markdown("\n**Indexed Documents:**"))
    for doc in kb_summary['documents']:
        print(f"  • {doc}")

---
## 💡 iPad & Google Cloud Tips

| Task | How |
|---|---|
| Upload document from iPad Photos | Cell 6 → tap upload widget → choose Photos |
| Upload from iPad Files app | Cell 6 → tap upload widget → choose Files |
| Process from Google Drive | Cell 13 → set `DRIVE_FILE_PATH` |
| View results on iPad | Google Drive app → `DocumentPipelineResults` folder |
| Run GCP setup | Google Cloud Shell at [console.cloud.google.com](https://console.cloud.google.com) |
| Document AI console | [console.cloud.google.com/ai/document-ai](https://console.cloud.google.com/ai/document-ai) |
| Vertex AI console | [console.cloud.google.com/vertex-ai](https://console.cloud.google.com/vertex-ai) |
| GitHub repo | [github.com/carlymariec/document-processing-pipeline](https://github.com/carlymariec/document-processing-pipeline) |

### North Dakota Civil Procedure Reference
- **NDRCP Rule 26** – General discovery obligations  
- **NDRCP Rule 26(b)(3)** – Attorney work-product protection  
- **NDRCP Rule 37** – Sanctions for failure to preserve  
- **NDCC § 31-09** – Admissibility of electronic records  

> ⚖️ *This tool is for informational purposes only and does not constitute legal advice.*